# 07 — Performance & Cost Tuning

## Overview

This lab teaches you how to measure, improve, and verify the performance of every Delta table in `lh_advanced_scenarios`. Small-file accumulation and unordered data layouts are the two most common causes of slow Fabric queries; this notebook addresses both systematically.**Safe defaults:** `p_dry_run = True` (no changes made until you explicitly set it to `False`).

**Prerequisites:** At least one of Scenarios 01–05 has been run so the target tables exist and contain data.  

**What you will learn**

- How to use `DESCRIBE DETAIL` to audit file counts and sizes before any changes- How to build a before/after comparison report so tuning gains are measurable and defensible

- How `OPTIMIZE` compacts many small Parquet files into right-sized 128 MB files — the single most impactful tuning for read latency- How to benchmark query latency before and after optimization with repeatable timing code

- How `ZORDER BY` co-locates rows by frequently-filtered columns so Spark can skip entire files using Delta's min/max statistics- How `VACUUM` reclaims storage by removing superseded file versions, and why the 168-hour retention floor must never be lowered in production

### Cell 1 — Runtime parameters
`p_retain_hours` controls how far back `VACUUM` looks when deleting obsolete Parquet files. The 168-hour floor (7 days) ensures that any concurrent reader, time-travel query, or Delta table clone that was started up to a week ago can still access the file versions it opened — lowering this risks `FileNotFoundException` in long-running queries. `p_dry_run = True` is a safety guard so you can review the SQL statements that **would** run without applying any changes; set it to `False` only when you are ready to commit.

### Cell 2 — Configuration: table list and ZORDER hints
`MANAGED_TABLES` enumerates every Delta table to profile and optimize — keeping it explicit avoids accidentally running `OPTIMIZE` on unrelated tables in the workspace. `ZORDER_COLS` maps each table to the column(s) most commonly used in `WHERE` filters or `JOIN` predicates. `ZORDER BY` reorders rows within each Parquet file so that related values are co-located; Delta then builds per-file min/max statistics ("data skipping") that allow the query engine to skip entire files when a filter value falls outside a file's range. Choosing the wrong columns wastes the ZORDER I/O without improving skipping — only high-cardinality, frequently-filtered columns benefit.

In [ ]:
p_retain_hours = 168    # VACUUM retention in hours (minimum 168 for safety)
p_dry_run      = True   # Set False to execute OPTIMIZE and VACUUM

In [ ]:
from pyspark.sql import functions as F
import time

LH = "lh_advanced_scenarios"
MANAGED_TABLES = [
    f"{LH}.bronze.orders_raw",
    f"{LH}.bronze.customers_cdc",
    f"{LH}.silver.orders",
    f"{LH}.silver.customers",
    f"{LH}.gold.dim_customer",
]

# ZORDER hints per table — choose columns most used in WHERE / JOIN predicates
ZORDER_COLS = {
    f"{LH}.silver.orders":      "order_date, status",
    f"{LH}.silver.customers":   "region, is_deleted",
    f"{LH}.gold.dim_customer":  "region, is_current",
}

## Phase A — Profile file sizes

`DESCRIBE DETAIL <table>` returns a single-row summary from Delta's transaction log without scanning any data files — it is near-instantaneous. The key metrics are `numFiles` (total active Parquet files) and `sizeInBytes` (total uncompressed logical size). Dividing the two gives `avg_file_mb`: the average file size. Delta's optimal target is **128 MB per file**. A table with 10,000 files averaging 50 KB is heavily fragmented and will suffer from Spark driver overhead, slow listing, and poor data-skipping. This baseline is what you will compare against in Phase E after `OPTIMIZE` runs.
### Cell 3 — Baseline file-size audit

In [ ]:
profile_rows = []
for tbl in MANAGED_TABLES:
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {tbl}").collect()[0]
        num_files  = detail["numFiles"]
        size_bytes = detail["sizeInBytes"]
        avg_mb     = round((size_bytes / num_files / 1024 / 1024), 2) if num_files > 0 else 0
        total_mb   = round(size_bytes / 1024 / 1024, 2)
        profile_rows.append((tbl, num_files, avg_mb, total_mb))
    except Exception as e:
        profile_rows.append((tbl, -1, -1, -1))
        print(f"Could not profile {tbl}: {e}")

profile_df = spark.createDataFrame(profile_rows, ["table_name", "num_files", "avg_file_mb", "total_mb"])
profile_df.show(truncate=False)

## Phase B — OPTIMIZE + ZORDER

`OPTIMIZE <table> ZORDER BY (<cols>)` does two things in one pass: (1) **compaction** — it rewrites all small files into right-sized 128 MB Parquet files, reducing file-listing and open/close overhead; (2) **Z-ordering** — it sorts data within files using a space-filling curve across the specified columns so that rows sharing similar key values are stored near each other. The result is that subsequent queries with `WHERE order_date = '...' AND status = '...'` touch a fraction of the files. The loop prints the SQL statement in dry-run mode so you can inspect it before committing. In live mode, `result.metrics` reports how many files were added (compacted targets) and removed (old small files), which is a direct measure of compaction effectiveness.
### Cell 4 — Run OPTIMIZE with optional ZORDER

In [ ]:
opt_results = []
for tbl in MANAGED_TABLES:
    zorder = ZORDER_COLS.get(tbl, "")
    zorder_clause = f"ZORDER BY ({zorder})" if zorder else ""
    sql = f"OPTIMIZE {tbl} {zorder_clause}"
    print(f"{'[DRY RUN] ' if p_dry_run else ''}Running: {sql}")
    if not p_dry_run:
        t0 = time.time()
        result = spark.sql(sql)
        elapsed = round(time.time() - t0, 1)
        metrics = result.select("metrics.*").collect()[0].asDict()
        opt_results.append((tbl, elapsed, metrics.get("numFilesAdded", 0), metrics.get("numFilesRemoved", 0)))
        print(f"  Done in {elapsed}s | +{metrics.get('numFilesAdded',0)} files / -{metrics.get('numFilesRemoved',0)} files")

if not p_dry_run and opt_results:
    spark.createDataFrame(opt_results, ["table","elapsed_s","files_added","files_removed"]).show(truncate=False)

## Phase C — VACUUM

`VACUUM <table> RETAIN <N> HOURS` physically deletes Parquet files that are no longer referenced by any Delta snapshot newer than `N` hours ago. Without regular VACUUM calls, every MERGE, OPTIMIZE, and overwrite operation accumulates orphaned files indefinitely — a busy production table can grow 5–10× larger than its logical size within a month. The assertion `p_retain_hours >= 168` is a safety net: Delta's default retention is 7 days (168 h) and concurrent readers, Delta Sharing consumers, and time-travel queries issued within that window will break if their referenced files are deleted prematurely. Only reduce the floor if you have verified no long-running queries will reference older snapshots.
### Cell 5 — Reclaim storage with VACUUM

In [ ]:
assert p_retain_hours >= 168, "Retention must be >= 168 hours to avoid breaking concurrent readers."

for tbl in MANAGED_TABLES:
    sql = f"VACUUM {tbl} RETAIN {p_retain_hours} HOURS"
    print(f"{'[DRY RUN] ' if p_dry_run else ''}Running: {sql}")
    if not p_dry_run:
        spark.sql(sql)
        print(f"  VACUUM complete for {tbl}")

## Phase D — Benchmark queries

Runs two representative queries — one aggregate over a date-range-filtered Silver table, one filtered scan of the Gold dimension — and records wall-clock latency using `time.time()`. `.collect()` forces a **full Spark action** (no lazy evaluation shortcut), so the elapsed time includes scheduling, data-skipping evaluation, file reads, shuffle, and results collection to the driver. Run this cell **before** OPTIMIZE (with `p_dry_run=True`) and again **after** (with `p_dry_run=False`) to quantify the improvement. The results are collected in `bench_results` and displayed as a DataFrame for easy copy-paste into stakeholder reports.
### Cell 6 — Time representative queries

In [ ]:
BENCHMARKS = [
    (
        "Silver orders agg by status (filtered date range)",
        f"SELECT status, COUNT(*) cnt, SUM(amount) rev FROM {LH}.silver.orders "
        f"WHERE order_date BETWEEN '2025-01-01' AND '2025-12-31' GROUP BY status"
    ),
    (
        "Gold dim_customer current rows by region",
        f"SELECT region, COUNT(*) cnt FROM {LH}.gold.dim_customer WHERE is_current = true GROUP BY region"
    ),
]

bench_results = []
for label, sql in BENCHMARKS:
    t0 = time.time()
    spark.sql(sql).collect()
    elapsed = round(time.time() - t0, 3)
    bench_results.append((label, elapsed))
    print(f"[{elapsed}s] {label}")

spark.createDataFrame(bench_results, ["benchmark", "elapsed_s"]).show(truncate=False)

## Phase E — Post-profile (compare with Phase A baseline)

Re-runs the same `DESCRIBE DETAIL` loop from Phase A and joins the results to the baseline snapshot captured in `profile_df`. Displaying `num_files` vs `num_files_after` and `avg_file_mb` vs `avg_file_mb_after` side-by-side gives a concrete, numeric proof of improvement: a table that went from 2,000 files averaging 0.4 MB to 12 files averaging 128 MB represents a 99.4% reduction in file-metadata overhead. This comparison should be saved and shared with stakeholders whenever justifying the compute cost of scheduled OPTIMIZE jobs.
### Cell 7 — Before/after comparison report

In [ ]:
post_rows = []
for tbl in MANAGED_TABLES:
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {tbl}").collect()[0]
        num_files  = detail["numFiles"]
        size_bytes = detail["sizeInBytes"]
        avg_mb = round((size_bytes / num_files / 1024 / 1024), 2) if num_files > 0 else 0
        post_rows.append((tbl, num_files, avg_mb))
    except Exception:
        post_rows.append((tbl, -1, -1))

post_df = spark.createDataFrame(post_rows, ["table_name", "num_files_after", "avg_file_mb_after"])
profile_df.join(post_df, "table_name") \
           .select("table_name", "num_files", "num_files_after", "avg_file_mb", "avg_file_mb_after") \
           .show(truncate=False)